### Task 1 

Create a folder containing at least five PDF documents. 

### Task 2 

Write a program to load all PDFs from the folder. 

### Task 3 

Split all documents into chunks and generate embeddings. 

### Task 4 

Store the chunks in ChromaDB and create a retriever.

### Task 5 


Ask questions that require information from different PDFs and verify that the 
chatbot retrieves the correct sources. 

In [1]:
#Load All PDFs from the Folder
import os 

from langchain_community.document_loaders import PyPDFLoader 

documents = [] 
folder = "Documents" 

for file in os.listdir(folder): 

    if file.endswith(".pdf"): 

        loader = PyPDFLoader(os.path.join(folder, file)) 

        documents.extend(loader.load()) 

print(f"Loaded {len(documents)} pages.") 

C:\Users\sachi\AppData\Local\Temp\ipykernel_25732\1044489116.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\sachi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 33 pages.


In [2]:
#Splitting Multiple Documents 

from langchain_text_splitters import RecursiveCharacterTextSplitter 

splitter = RecursiveCharacterTextSplitter( 

    chunk_size=500, 

    chunk_overlap=100 

) 
chunks = splitter.split_documents(documents) 

print(len(chunks)) 

151


In [3]:
# Load the embedding model
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3300.20it/s]


Embeddings model loaded successfully.


In [4]:
#Store Everything in ChromaDB
from langchain_chroma import Chroma 
vector_db = Chroma.from_documents( 

    documents=chunks, 

    embedding=embedding, 

    persist_directory="./chroma_db" 

) 

In [5]:
# Create a retriever for similarity search
retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}   # Retrieve the top 3 relevant chunks
)

print("Vector database and retriever created successfully.")

Vector database and retriever created successfully.


In [8]:
# Ask a question
question = input("Enter query")

# Retrieve the most relevant document chunks
docs = retriever.invoke(question)

print("Question:")
print(question)

print("\nRetrieved Sources:\n")

# Display the retrieved source information
for doc in docs:
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:200])   # Print the first 200 characters
    print("-" * 50)

Question:
cat eligibility

Retrieved Sources:

Source: Documents\CAT_2026_Eligibility_26-07-26.pdf
Page: 0
Common Admission Test (CAT) 2026 
Eligibility    
● The candidate must hold a Bachelor’s Degree, with at least 50% marks or equivalent CGPA [45% in the 
case of candidates belonging to the Scheduled C
--------------------------------------------------
Source: Documents\CAT_2026_Information_Bulletin_26-07-26.pdf
Page: 2
INFORMATION SOURCES   
CAT website: https://iimcat.ac.in  Help Desk Number: 1800 210 0175  
CAT 2026 ELIGIBILITY   
Eligibility    
● The candidate must hold a Bachelor’s Degree, with at least 50% mar
--------------------------------------------------
Source: Documents\CAT_2026_Information_Bulletin_26-07-26.pdf
Page: 4
REGISTRATION FOR CAT   
The CAT 2026 registration window opens at 10:00 a.m. on August 3, 2026 and will close at 5:00 p.m. on 
September 15, 2026. The details will be available in the registration gui
--------------------------------------------------